# Semana 05: Orquestração de Dados e Fluxos IIoT com Node-RED

## Laboratório Prático: 3 Contêineres Docker (Mosquitto + Node-RED + Flask Monolítico) e Observabilidade via Painel Debug

Este notebook apresenta a fundamentação teórica e o roteiro prático da **Semana 05** da disciplina de **Automação Industrial**. Toda a infraestrutura da aula roda de forma **100% conteinerizada via Docker** (composta por 3 contêineres: Broker Mosquitto MQTT, Node-RED e a Aplicação Monolítica Flask que atua como simulador do chão de fábrica).

### Objetivos de Aprendizagem
- Compreender a filosofia de **Flow-Based Programming (FBP)** e o ecossistema do **Node-RED** na Indústria 4.0.
- Subir e orquestrar a infraestrutura completa de 3 contêineres com **Docker Compose** (`mosquitto`, `nodered`, `flask_app`).
- Compreender o funcionamento do simulador monolítico **Flask** como gerador de eventos e gateway MQTT na rede industrial.
- Construir, importar e configurar fluxos de ingestão com nós de entrada (`mqtt in`), conversão (`json`), roteamento (`switch`), tratamento lógico (`function`) e observabilidade (`debug`).
- Simular anomalias industriais (superaquecimento, vibração crítica, parada de emergência e contagem de peças) e inspecionar os pacotes em tempo real no **Painel Debug** do Node-RED.

---

## 1. Fundamentação Teórica: Flow-Based Programming & Node-RED

### 1.1 O que é Programação Baseada em Fluxos (FBP)?
A **Programação Baseada em Fluxos** (*Flow-Based Programming*), concebida por J. Paul Morrison na década de 1970, é um paradigma no qual as aplicações são modeladas como redes de processos de "caixa-preta" (*nós*) que trocam dados através de conexões de passagem de mensagens predefinidas.

Diferente da programação imperativa tradicional — onde o fluxo de controle é sequencial e guiado por instruções de código —, no FBP a execução é orientada a **eventos e dados** (*Data-Driven*). Cada nó processa as mensagens recebidas em suas entradas (*inputs*), aplica transformações e as emite em suas saídas (*outputs*).

```
 ┌──────────────┐   msg.payload    ┌──────────────┐   msg.payload    ┌──────────────┐
 │  Nó Entrada  │ ──────────────── │ Nó Tratamento│ ──────────────── │  Nó de Saída │
 │ (MQTT / CLP) │                  │  (Function)  │                  │(Debug/Influx)│
 └──────────────┘                  └──────────────┘                  └──────────────┘
```

### 1.2 O Ecossistema do Node-RED na IIoT
Criado na **IBM Emerging Technology Services** em 2013 e mantido pela **OpenJS Foundation**, o Node-RED é a ferramenta de orquestração visual (*Low-Code*) de referência para a automação e IIoT.

Construído sobre o runtime assíncrono do **Node.js**, o Node-RED oferece:
1. **Editor Web Gráfico:** Desenvolvimento acessível via navegador web (porta padrão `1880`).
2. **Conectividade Industrial Nativa:** Suporte a MQTT, Modbus, OPC UA, HTTP REST, WebSockets e bancos temporais (InfluxDB).
3. **Manipulação de Objetos JavaScript:** As mensagens trafegam na forma de um objeto `msg`, contendo `msg.topic` e `msg.payload`.

### 1.3 Principais Categorias de Nós no Node-RED

| Categoria | Nós Principais | Função no Pipeline IIoT |
| :--- | :--- | :--- |
| **Entrada (Input)** | `inject`, `mqtt in`, `http in` | Injetam eventos ou capturam telemetria do chão de fábrica. |
| **Processamento (Function)** | `function`, `switch`, `change`, `json` | Aplicam lógica em JavaScript, roteamento condicional e conversão de formato. |
| **Saída (Output)** | `debug`, `mqtt out`, `http response` | Exibem dados na aba lateral de Debug, publicam de volta no broker ou enviam a atuadores. |

---

## 2. Arquitetura dos 3 Contêineres Docker

Toda a prática está encapsulada dentro da pasta `aulas/semana_05/`, rodando em 3 contêineres conectados pela rede interna `rede_automacao`:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      REDE DOCKER (rede_automacao)                       │
│                                                                         │
│  ┌───────────────────────┐             ┌─────────────────────────────┐  │
│  │   SIMULADOR FLASK     │             │   ECLIPSE MOSQUITTO         │  │
│  │  (flask_simulator_app)│ ──────────► │   (mosquitto_broker)        │  │
│  │   http://localhost    │ MQTT :1883  │   Porta TCP: 1883           │  │
│  │        :5000          │             │   Porta WS:  9001           │  │
│  └───────────────────────┘             └──────────────┬──────────────┘  │
│                                                       │                 │
│                                                       │ Subscrição      │
│                                                       │ fabrica/#       │
│                                                       ▼                 │
│                                        ┌─────────────────────────────┐  │
│                                        │          NODE-RED           │  │
│                                        │        (nodered_app)        │  │
│                                        │   http://localhost:1880     │  │
│                                        │  [PAINEL LATERAL DEBUG 🪲]  │  │
│                                        └─────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────────────┘
```

1. **`mosquitto_broker`:** Broker MQTT leve na porta `1883`.
2. **`nodered_app`:** Orquestrador de fluxos na porta `1880`.
3. **`flask_simulator_app`:** Simulador da planta fabril com painel SCADA na porta `5000`.

---

## 3. Subindo os 3 Contêineres com Docker Compose

Para subir os 3 serviços de uma só vez, abra o terminal e execute:

```bash
cd aulas/semana_05
docker compose up --build -d
```

Para verificar se os 3 contêineres estão operacionais:
```bash
docker compose ps
```

In [ ]:
# Verificação de Conectividade com os 3 Contêineres Docker (Portas 1883, 1880 e 5000)
import socket

def test_port(host, port, service_name):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2.0)
    try:
        s.connect((host, port))
        print(f"✅ {service_name} está ATIVO e acessível em {host}:{port}")
        s.close()
        return True
    except Exception as e:
        print(f"❌ {service_name} NÃO acessível em {host}:{port} ({e})")
        print(f"   -> Execute 'docker compose up --build -d' na pasta aulas/semana_05")
        return False

print("--- TESTANDO STATUS DOS 3 CONTÊINERES DOCKER ---")
mqtt_ok = test_port("localhost", 1883, "1. Broker Eclipse Mosquitto (MQTT)")
nodered_ok = test_port("localhost", 1880, "2. Servidor Node-RED")
flask_ok = test_port("localhost", 5000, "3. Simulador Monolítico Flask")


--- 

## 4. Consultando o Simulador Flask da Fábrica Virtual

Acesse a interface gráfica SCADA no navegador: **[http://localhost:5000](http://localhost:5000)**.

Você também pode interagir com o contêiner Flask diretamente via código Python no notebook:

In [ ]:
# Consulta de Telemetria e Status da Planta via API REST do Flask
import urllib.request
import json

FLASK_API_URL = "http://localhost:5000/api/status"

try:
    req = urllib.request.Request(FLASK_API_URL)
    with urllib.request.urlopen(req, timeout=3) as response:
        if response.status == 200:
            data = json.loads(response.read().decode('utf-8'))
            print("✅ Conexão com o Contêiner Flask bem-sucedida!")
            print("\n--- ESTADO ATUAL DA PLANTA VIRTUAL ---")
            print(f"• Broker MQTT Conectado: {data.get('broker_connected')}")
            print(f"• Simulação Ativa: {data.get('simulation_running')}")
            print(f"• Intervalo de Envio: {data.get('publish_interval')}s")
            print(f"• Total de Mensagens Enviadas: {data.get('total_messages_sent')}")
            print(f"• Telemetria da Máquina: {data.get('machine_state')}")
            print(f"• Indicadores de Produção: {data.get('production_state')}")
except Exception as e:
    print(f"⚠️ Não foi possível consultar a API do Flask em {FLASK_API_URL} ({e})")
    print("   -> Verifique se o contêiner 'flask_simulator_app' está rodando no Docker")


---

## 5. Construção e Importação do Fluxo no Node-RED

O arquivo [`flows_semana05.json`](aulas/semana_05/flows_semana05.json) contém o fluxo pronto para o laboratório.

### 5.1 Importando o Fluxo no Node-RED
1. Abra o Node-RED: **[http://localhost:1880](http://localhost:1880)**
2. Pressione `Ctrl + I` (ou Menu ☰ > **Import**).
3. Cole o conteúdo de `flows_semana05.json` na caixa de texto.
4. Clique em **Import** e depois no botão vermelho **Deploy** (canto superior direito).
5. Abra o **Painel lateral Debug** (ícone do inseto `🪲` no canto superior direito).

### 5.2 Estrutura do Pipeline de Dados

```
 [MQTT In: fabrica/#] ───► [JSON Parser] ───► [Switch: Roteador de Tópicos]
                                                    │
                     ┌──────────────────────────────┼──────────────────────────────┐
                     │ (se 'telemetria')            │ (se 'alarmes')               │ (se 'producao')
                     ▼                              ▼                              ▼
            [Function: Limites]           [Function: Prioridade]         [Function: OEE & Taxa]
                     │                              │                              │
                     ▼                              ▼                              ▼
            [Debug: 🟢 Telemetria]         [Debug: 🔴 Alarmes]            [Debug: 📦 Produção]
```

1. **`MQTT In (fabrica/#)`:** Assina todos os subtópicos da fábrica.
2. **`JSON Parser`:** Converte payload em objeto JS.
3. **`Switch`:** Roteia para os tratamentos adequados.
4. **`Function`:** Classifica severidade (Normal, Alerta, Crítico) e calcula taxas de qualidade.
5. **`Debug`:** Exibe as saídas organizadas por cores e categorias na aba lateral.

---

## 6. Prática Hands-On: Disparo de Cenários e Observação no Debug

Podemos disparar eventos industriais através da interface web do Flask ([http://localhost:5000](http://localhost:5000)) ou executando a célula abaixo para simular chamadas via API REST:

Abra a aba de **Debug** no Node-RED (`🪲`) para acompanhar os resultados!

In [ ]:
# Disparo Programático de Cenários Críticos para Observação no Debug do Node-RED
import urllib.request
import json
import time

def trigger_flask_scenario(scenario_name, label):
    url = "http://localhost:5000/api/simulator/event"
    data = json.dumps({"scenario": scenario_name}).encode('utf-8')
    req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=3) as resp:
            print(f"🚀 Cenário disparado com sucesso: [{label}]")
    except Exception as e:
        print(f"⚠️ Erro ao disparar cenário {scenario_name}: {e}")

print("--- DISPARANDO BATERIA DE EVENTOS CRÍTICOS ---\n")

# 1. Disparar Superaquecimento
trigger_flask_scenario("high_temp", "🔥 Superaquecimento na Prensa CNC (>88°C)")
time.sleep(2.0)

# 2. Disparar Vibração Excessiva
trigger_flask_scenario("high_vib", "⚡ Vibração Mecânica Crítica (9.45 mm/s)")
time.sleep(2.0)

# 3. Disparar Parada de Emergência
trigger_flask_scenario("emergency_stop", "🛑 DISPARO DE PARADA DE EMERGÊNCIA (E-STOP)")
time.sleep(2.0)

# 4. Resetar Linha para Operação Normal
trigger_flask_scenario("reset_normal", "🔄 Reset e Restabelecimento de Operação Normal")

print("\n✨ Eventos enviados! Inspecione a saída formatada no Painel Debug do Node-RED (:1880).")


---

## 7. Exercícios de Fixação e Avaliação

### Questão 1 (Arquitetura Docker & FBP)
Explique por que isolar o Broker MQTT, o Node-RED e a aplicação Flask em 3 contêineres Docker independentes conectados em rede bridge representa uma boa prática de engenharia de software e segurança na arquitetura IIoT.

### Questão 2 (Roteamento no Nó Switch)
No Node-RED, como o nó `switch` utiliza a propriedade `msg.topic` para segregar fluxos de alarmes emergenciais de leituras periódicas de telemetria?

### Questão 3 (Manipulação no Nó Function)
Considere que um novo sensor de pressão hidráulica publica mensagens no tópico `fabrica/linha1/prensa01/pressao` com o seguinte payload:
```json
{
  "sensor_id": "PT_101",
  "pressao_bar": 8.4,
  "limite_max": 7.0
}
```
Escreva o código JavaScript para um nó `function` do Node-RED que:
1. Verifique se `pressao_bar > limite_max`.
2. Caso positivo, crie a propriedade `msg.alerta_pressao = "SOBREPRESSAO_CRITICA"`.
3. Caso negativo, defina `msg.alerta_pressao = "OK"`.
4. Retorne o objeto `msg` modificado.

### Desafio Prático de Laboratório
1. No painel web do Flask ([http://localhost:5000](http://localhost:5000)), utilize o formulário **Publicador Manual MQTT** para enviar um payload customizado com suas iniciais.
2. Localize a mensagem no nó `🔍 [DEBUG] Todas Msg Brutas` na aba Debug do Node-RED.
3. Capture a tela comprovando o recebimento da mensagem no Node-RED.